In [ ]:
pip install open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model and preprocess function

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

### Prepare the augmentation pipeline

In [ ]:
from datasets import load_dataset

ds = load_dataset("axiong/imagenet-r")

In [ ]:
print(ds['test'][0]['image'])

In [ ]:
augment_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)) # from clip
])

In [ ]:
def generate_N_views(N, transform_fn, image):
  views = [transform_fn(image) for _ in range(N)]
  curr_img = preprocess(image)
  views.append(curr_img)
  views = torch.stack(views)

  return views

In [ ]:
sample_views_for_idx_0 = generate_N_views(63, augment_transform, ds['test'][0]['image'])

In [29]:
sample_views_for_idx_0.shape

torch.Size([64, 3, 224, 224])